In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from textblob import TextBlob
from collections import Counter
from nltk import word_tokenize, pos_tag
import requests
import time
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import pickle

In [ ]:
NOTEBOOK_DIR = Path.cwd().resolve()
_candidates = [
    NOTEBOOK_DIR / "data" / "moltbook_posts_preprocessed.parquet",
    NOTEBOOK_DIR.parent / "data" / "moltbook_posts_preprocessed.parquet",
]
PREP_PATH = next((p for p in _candidates if p.exists()), None)
if PREP_PATH is None:
    raise FileNotFoundError(
        "Run scripts/preprocess_engagement.py first."
    )

DATA_DIR = PREP_PATH.parent
FEATURES_DIR = DATA_DIR / "features"
FEATURES_DIR.mkdir(exist_ok=True)

df = pd.read_parquet(PREP_PATH)
print(f"Loaded {len(df):,} rows")
print(f"Columns: {list(df.columns)}")

Loaded 44,376 rows from /Users/neil/Desktop/CS4120_NLP/CS4120_moltbook_sentiment_analysis/data/moltbook_posts_preprocessed.parquet
Columns: ['annotation_row_id', 'topic_label', 'toxic_level', 'post_id', 'title', 'content', 'created_at', 'comment_count', 'upvotes', 'downvotes', 'submolt_id', 'submolt_name', 'submolt_display_name', 'has_url', 'title_char_len', 'content_char_len', 'combined_text', 'combined_char_len', 'title_word_count', 'content_word_count', 'combined_word_count', 'sentence_count', 'hour_of_day', 'day_of_week', 'engagement_score', 'engagement_class']


In [ ]:
# Sentiment features (VADER + TextBlob)
sid = SentimentIntensityAnalyzer()

texts = df["combined_text"].fillna("").tolist()

vader_scores = [sid.polarity_scores(t) for t in texts]
df["vader_compound"] = [s["compound"] for s in vader_scores]

tb_results = [TextBlob(t).sentiment for t in texts]
df["tb_polarity"] = [s.polarity for s in tb_results]
df["tb_subjectivity"] = [s.subjectivity for s in tb_results]

df[["vader_compound", "tb_polarity", "tb_subjectivity"]].describe().round(4)

Computing VADER compound scores...


Computing TextBlob polarity/subjectivity...


Done.


,vader_compound,tb_polarity,tb_subjectivity
count,44376.0000,44376.0000,44376.0000
mean,0.4594,0.1590,0.4404
std,0.5663,0.2197,0.2299
min,-1.0000,-1.0000,0.0000
25%,0.0000,0.0050,0.3383
50%,0.7339,0.1280,0.4554
75%,0.9136,0.2667,0.5889
max,1.0000,1.0000,1.0000


In [ ]:
# Part-of-speech counts
POS_GROUPS = {
    "noun_count": {"NN", "NNS", "NNP", "NNPS"},
    "pronoun_count": {"PRP", "PRP$", "WP", "WP$"},
    "verb_count": {"VB", "VBD", "VBG", "VBN", "VBP", "VBZ"},
    "adjective_count": {"JJ", "JJR", "JJS"},
    "adverb_count": {"RB", "RBR", "RBS"},
}

pos_records = []
for i, text in enumerate(texts):
    tags = pos_tag(word_tokenize(text))
    tag_counts = Counter(tag for _, tag in tags)
    pos_records.append({
        group: sum(tag_counts.get(t, 0) for t in tagset)
        for group, tagset in POS_GROUPS.items()
    })


pos_df = pd.DataFrame(pos_records)
for col in pos_df.columns:
    df[col] = pos_df[col].values

df[list(POS_GROUPS.keys())].describe().round(2)

Computing POS tag counts...



   5,000/44,376


  10,000/44,376


  15,000/44,376


  20,000/44,376

...


  35,000/44,376


  40,000/44,376


  44,376/44,376


Done.


,noun_count,pronoun_count,verb_count,adjective_count,adverb_count
count,44376.00,44376.00,44376.00,44376.00,44376.00
mean,45.68,9.07,21.57,12.44,5.74
std,90.67,16.10,35.27,35.40,9.97
min,0.00,0.00,0.00,0.00,0.00
25%,13.00,0.00,4.00,3.00,1.00
50%,26.00,4.00,12.00,7.00,3.00
75%,56.00,12.00,29.00,15.00,8.00
max,8344.00,1582.00,2411.00,5556.00,701.00


In [ ]:
"""Text embeddings (Nomic-Embed-Text via Ollama)

Embeds each post's combined text into a 768-dim vector.  
**Strategy:** send full text first. If Ollama rejects it (context length exceeded),
binary-search the max char count that fits, so we keep as much content as possible.  
Results are cached to `data/features/embeddings.npy`."""

"""EMBEDDING_CACHE = FEATURES_DIR / "embeddings.npy"
OLLAMA_URL = "http://localhost:11434/api/embed"
MODEL = "nomic-embed-text"
BATCH_SIZE = 32
EMBED_DIM = 768


def _safe_text(text: str) -> str:
    #Ensure non-empty input for the embedding API.
    if not text or not text.strip():
        return "empty"
    return text


def embed_batch(texts: list[str]) -> np.ndarray:
    #Embed a batch of texts. Raises on API error.
    cleaned = [_safe_text(t) for t in texts]
    resp = requests.post(
        OLLAMA_URL,
        json={"model": MODEL, "input": cleaned},
        timeout=120,
    )
    resp.raise_for_status()
    return np.array(resp.json()["embeddings"], dtype=np.float32)


def embed_single_maxfit(text: str) -> tuple[np.ndarray, int]:
    
    t = _safe_text(text)
    # try full text first
    try:
        resp = requests.post(
            OLLAMA_URL,
            json={"model": MODEL, "input": [t]},
            timeout=60,
        )
        resp.raise_for_status()
        return np.array(resp.json()["embeddings"][0], dtype=np.float32), len(t)
    except requests.exceptions.HTTPError:
        pass  # too long, binary-search below

    # binary search: find max chars that the API accepts
    lo, hi = 500, len(t)
    while lo < hi - 1:
        mid = (lo + hi) // 2
        try:
            resp = requests.post(
                OLLAMA_URL,
                json={"model": MODEL, "input": [t[:mid]]},
                timeout=60,
            )
            resp.raise_for_status()
            lo = mid
        except requests.exceptions.HTTPError:
            hi = mid

    resp = requests.post(
        OLLAMA_URL,
        json={"model": MODEL, "input": [t[:lo]]},
        timeout=60,
    )
    resp.raise_for_status()
    return np.array(resp.json()["embeddings"][0], dtype=np.float32), lo


if EMBEDDING_CACHE.exists():
    embeddings = np.load(EMBEDDING_CACHE)
else:
    all_texts = df["combined_text"].fillna("").tolist()
    n = len(all_texts)
    all_embeddings = np.zeros((n, EMBED_DIM), dtype=np.float32)
    truncated_posts = []  # track which posts needed truncation
    t0 = time.time()
    done = 0

    start = 0
    while start < n:
        end = min(start + BATCH_SIZE, n)
        batch = all_texts[start:end]

        try:
            # try the whole batch with full texts
            all_embeddings[start:end] = embed_batch(batch)
            done = end
        except requests.exceptions.HTTPError:
            # at least one text in this batch is too long;
            # fall back to one-by-one with binary-search truncation
            for i, text in enumerate(batch):
                idx = start + i
                emb, chars_used = embed_single_maxfit(text)
                all_embeddings[idx] = emb
                if chars_used < len(text):
                    truncated_posts.append(
                        (idx, len(text), chars_used)
                    )
            done = end

        elapsed = time.time() - t0
        rate = done / elapsed if elapsed > 0 else 0
        eta = (n - done) / rate if rate > 0 else 0
        print(
            f"\r  {done:>6,}/{n:,}  "
            f"({done/n*100:5.1f}%)  "
            f"{rate:.0f} posts/s  "
            f"ETA {eta:.0f}s  "
            f"trunc={len(truncated_posts)}",
            end="", flush=True,
        )
        start = end

    embeddings = all_embeddings
    np.save(EMBEDDING_CACHE, embeddings)
    elapsed_total = time.time() - t0
    print(f"\nDone in {elapsed_total:.0f}s. Saved {embeddings.shape} -> {EMBEDDING_CACHE}")

    if truncated_posts:
        total_orig = sum(o for _, o, _ in truncated_posts)
        total_kept = sum(k for _, _, k in truncated_posts)
        print(f"\nTruncated {len(truncated_posts)} posts ({len(truncated_posts)/n*100:.2f}%):")
        print(f"  Total chars in truncated posts: {total_orig:,} -> {total_kept:,} ({total_kept/total_orig*100:.1f}% kept)")
        all_chars = sum(len(t) for t in all_texts)
        lost = total_orig - total_kept
        print(f"  Overall text preserved: {(all_chars - lost)/all_chars*100:.2f}%")
        print(f"\n  Worst truncations:")
        for idx, orig, kept in sorted(truncated_posts, key=lambda x: x[2]/x[1])[:10]:
            print(f"    idx={idx}: {orig:>7,} -> {kept:>7,} chars ({kept/orig*100:.1f}% kept)")
    else:
        print("No posts needed truncation.")

assert embeddings.shape == (len(df), EMBED_DIM), (
    f"Embedding shape {embeddings.shape} != expected ({len(df)}, {EMBED_DIM})"
)
print(f"\nEmbedding matrix: {embeddings.shape}  dtype={embeddings.dtype}")"""

Loaded cached embeddings: (44376, 768)

Embedding matrix: (44376, 768)  dtype=float32


In [ ]:
# Metadata / engineered features
# one-hot: topic_label (9 categories A-I)
topic_ohe = pd.get_dummies(df["topic_label"], prefix="topic").astype(np.float32)

# one-hot: toxic_level (0-4)
toxic_ohe = pd.get_dummies(df["toxic_level"], prefix="toxic").astype(np.float32)

# numeric features (will be standardized after split)
numeric_cols = [
    "combined_word_count",
    "sentence_count",
    "title_char_len",
    "content_char_len",
    "has_url",
    "hour_of_day",
    "day_of_week",
    "vader_compound",
    "tb_polarity",
    "tb_subjectivity",
    "noun_count",
    "pronoun_count",
    "verb_count",
    "adjective_count",
    "adverb_count",
]
numeric_raw = df[numeric_cols].values.astype(np.float32)

print(f"topic_ohe:     {topic_ohe.shape}")
print(f"toxic_ohe:     {toxic_ohe.shape}")
print(f"numeric_raw:   {numeric_raw.shape}")
print(f"embeddings:    {embeddings.shape}")

topic_ohe:     (44376, 9)
toxic_ohe:     (44376, 5)
numeric_raw:   (44376, 15)
embeddings:    (44376, 768)


In [ ]:
# Assemble full feature matrix
feature_blocks = {
    "embeddings": embeddings,
    "topic_ohe": topic_ohe.values,
    "toxic_ohe": toxic_ohe.values,
    "numeric": numeric_raw,
}

offset = 0
feature_index = {}
for name, block in feature_blocks.items():
    w = block.shape[1]
    feature_index[name] = (offset, offset + w)
    offset += w

X_all = np.hstack([b for b in feature_blocks.values()]).astype(np.float32)

label_map = {"lower": 0, "lower_middle": 1, "upper_middle": 2, "upper": 3}
label_map_inv = {v: k for k, v in label_map.items()}
y_all = df["engagement_class"].map(label_map).values.astype(np.int64)

print(f"X_all: {X_all.shape}  (features per post)")
print(f"y_all: {y_all.shape}  unique={np.unique(y_all)}")

X_all: (44376, 797)  (features per post)
y_all: (44376,)  unique=[0 1 2 3]

Feature block index (col start:end):
       embeddings: cols [0:768)  (768 dims)
        topic_ohe: cols [768:777)  (9 dims)
        toxic_ohe: cols [777:782)  (5 dims)
          numeric: cols [782:797)  (15 dims)


In [ ]:
# Stratified train / val / test split (70 / 15 / 15)
SEED = 42

X_train, X_temp, y_train, y_temp, idx_train, idx_temp = train_test_split(
    X_all, y_all, np.arange(len(df)),
    test_size=0.30, stratify=y_all, random_state=SEED,
)
X_val, X_test, y_val, y_test, idx_val, idx_test = train_test_split(
    X_temp, y_temp, idx_temp,
    test_size=0.50, stratify=y_temp, random_state=SEED,
)

label_map_inv = {v: k for k, v in label_map.items()}


for name, y_split in [("Train", y_train), ("Val", y_val), ("Test", y_test)]:
    counts = np.bincount(y_split, minlength=4)
    pcts = counts / counts.sum() * 100
    parts = "  ".join(f"{label_map_inv[i]}={c}({p:.1f}%)"
                      for i, (c, p) in enumerate(zip(counts, pcts)))
    print(f"{name:>5s}: {parts}")

Train: 31,063  (70.0%)
Val:   6,656  (15.0%)
Test:  6,657  (15.0%)

Train: lower=7759(25.0%)  lower_middle=8109(26.1%)  upper_middle=7481(24.1%)  upper=7714(24.8%)
  Val: lower=1663(25.0%)  lower_middle=1737(26.1%)  upper_middle=1603(24.1%)  upper=1653(24.8%)
 Test: lower=1663(25.0%)  lower_middle=1738(26.1%)  upper_middle=1603(24.1%)  upper=1653(24.8%)


In [ ]:
# Normalize numeric features (fit on train only)
num_start, num_end = feature_index["numeric"]

scaler = StandardScaler()
X_train[:, num_start:num_end] = scaler.fit_transform(X_train[:, num_start:num_end])
X_val[:, num_start:num_end] = scaler.transform(X_val[:, num_start:num_end])
X_test[:, num_start:num_end] = scaler.transform(X_test[:, num_start:num_end])

Scaled numeric features (cols 782:797)
Train numeric mean: [-0.  0. -0.  0.  0. -0.  0.  0. -0.  0. -0. -0.  0. -0.  0.]
Train numeric std:  [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [ ]:
np.save(FEATURES_DIR / "X_train.npy", X_train)
np.save(FEATURES_DIR / "X_val.npy",   X_val)
np.save(FEATURES_DIR / "X_test.npy",  X_test)
np.save(FEATURES_DIR / "y_train.npy", y_train)
np.save(FEATURES_DIR / "y_val.npy",   y_val)
np.save(FEATURES_DIR / "y_test.npy",  y_test)
np.save(FEATURES_DIR / "idx_train.npy", idx_train)
np.save(FEATURES_DIR / "idx_val.npy",   idx_val)
np.save(FEATURES_DIR / "idx_test.npy",  idx_test)

meta = {
    "feature_index": feature_index,
    "label_map": label_map,
    "label_map_inv": label_map_inv,
    "numeric_cols": numeric_cols,
    "scaler": scaler,
    "seed": SEED,
    "n_train": len(y_train),
    "n_val": len(y_val),
    "n_test": len(y_test),
    "total_features": X_all.shape[1],
}
with open(FEATURES_DIR / "meta.pkl", "wb") as f:
    pickle.dump(meta, f)

df.to_parquet(DATA_DIR / "moltbook_posts_with_sentiment.parquet", index=False)

print("Saved to", FEATURES_DIR)
for p in sorted(FEATURES_DIR.iterdir()):
    size_mb = p.stat().st_size / 1024 / 1024
    print(f"  {p.name:>20s}  {size_mb:6.1f} MB")

Saved to /Users/neil/Desktop/CS4120_NLP/CS4120_moltbook_sentiment_analysis/data/features
            X_test.npy    20.2 MB
           X_train.npy    94.4 MB
             X_val.npy    20.2 MB
        embeddings.npy   130.0 MB
          idx_test.npy     0.1 MB
         idx_train.npy     0.2 MB
           idx_val.npy     0.1 MB
              meta.pkl     0.0 MB
            y_test.npy     0.1 MB
           y_train.npy     0.2 MB
             y_val.npy     0.1 MB
